In [0]:
%pip install aiohttp
%restart_python

In [0]:
import asyncio
import aiohttp
from azure.storage.blob.aio import BlobServiceClient
import json 
import io
import requests
from databricks.sdk import WorkspaceClient

In [0]:
dbutils.widgets.text("scope_name", "default2")
SECRET_SCOPE = dbutils.widgets.get("scope_name")

In [0]:
storage_conn_string = dbutils.secrets.get(scope=SECRET_SCOPE, key="storage-conn-str")

In [0]:
async def run(session, url):
    blob_service_client = BlobServiceClient.from_connection_string(storage_conn_string)
    container_client = blob_service_client.get_container_client("lechster10")

    lines = []
    file_number = 1

    async with session.get(url) as response:
        async for line in response.content:
            decoded_line = line.decode('utf-8').strip()
            lines.append(decoded_line)            

            if len(lines) >= 50:            # every 50 lines we save a file
                content = "\n".join(lines)
                blob_name = f"data_{file_number}.jsonl"
                await container_client.upload_blob(name=f"streaming_crime_incidents/{blob_name}", data=content, overwrite=True)
                print(f"Saved file {blob_name}, number of lines: {len(lines)}")

                lines = []
                file_number += 1

    
    if lines:
        content = "\n".join(lines)
        blob_name = f"data_{file_number}"
        await container_client.upload_blob(name=f"streaming_crime_incidents/{blob_name}.jsonl", data=content, overwrite=True)
        print(f"Saved file {blob_name}, number of lines: {len(lines)}")

In [0]:
w = WorkspaceClient()
app_client_id = w.apps.get("csv-api-app").oauth2_app_client_id

In [0]:
url = "https://adb-7405604503619901.1.azuredatabricks.net/oidc/v1/token"

notebook_token = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().get()
)

data = {
    "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
    "subject_token": notebook_token,
    "subject_token_type": "urn:databricks:params:oauth:token-type:personal-access-token",
    "requested_token_type": "urn:ietf:params:oauth:token-type:access_token",
    "scope": "all-apis",
    "audience": app_client_id,
}

response = requests.post(url=url, data=data)
audience_token = response.json()["access_token"]

In [0]:
headers = {"Authorization": f"Bearer {audience_token}"}
custom_timeout = aiohttp.ClientTimeout(total=None)

In [0]:
URL = "https://csv-api-app-7405604503619901.1.azure.databricksapps.com/api/stream" 

async with aiohttp.ClientSession(headers=headers, timeout = custom_timeout) as session:
    await run(session, URL) 